In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error
from tqdm import tqdm
from model import prediction_process

In [ ]:
# working_dir = "/kaggle/input/rohlik-orders-forecasting-challenge"
working_dir = "data/"

try:
    train = pd.read_csv(f"{working_dir}train.csv")
    train_calendar = pd.read_csv(f"{working_dir}train_calendar.csv")
    test = pd.read_csv(f"{working_dir}test.csv")
    test_calendar = pd.read_csv(f"{working_dir}test_calendar.csv")
except FileNotFoundError:
    dotenv.load_dotenv(dotenv.find_dotenv())
    kagglehub.login()
    kagglehub.competition_download("rohlik-orders-forecasting-challenge", output_dir = "./data")
    train = pd.read_csv(f"{working_dir}train.csv")
    train_calendar = pd.read_csv(f"{working_dir}train_calendar.csv")
    test = pd.read_csv(f"{working_dir}test.csv")
    test_calendar = pd.read_csv(f"{working_dir}test_calendar.csv")

train["date"] = pd.to_datetime(train["date"])
train = train.sort_values(by=["warehouse", "date"])
train_calendar["date"] = pd.to_datetime(train_calendar["date"])
train_calendar = train_calendar.sort_values(by=["warehouse", "date"])
test = test.sort_values(by=["warehouse", "date"])
test_calendar = test_calendar.sort_values(by=["warehouse", "date"])

In [6]:
# train = train[["date","orders"]].rename(columns={"orders":"target", "date":"timestamp"})
train = train.rename(columns={"orders":"target", "date":"timestamp"})
# test["orders"] = pd.NA
# test["orders"] = test["orders"].astype(float)
test = test.reindex(columns=test.columns.tolist() + ["orders"])
# test = test[["date", "orders"]].rename(columns={"orders":"target", "date":"timestamp"})
test = test.rename(columns={"orders":"target", "date":"timestamp"})

In [8]:
full_predictions = pd.DataFrame()
for warehouse in tqdm(train["warehouse"].unique()):
    train_warehouse = train[train["warehouse"]==warehouse].copy(deep=True)
    test_warehouse = test[test["warehouse"]==warehouse].copy(deep=True)
    prediction_horizon = (max(test_warehouse["timestamp"]) - min(test_warehouse["timestamp"])).days + 1
    timestep = prediction_horizon
    predictions = prediction_process(train_warehouse, test_warehouse,
                                     prediction_horizon=prediction_horizon, timestep=timestep,
                                     freq="D")
    predictions["warehouse"] = warehouse
    full_predictions = pd.concat([full_predictions, predictions]) if not full_predictions.empty\
                                                                  else predictions
full_predictions

100%|██████████| 7/7 [00:00<00:00,  7.63it/s]


,timestamp,date_cut,predictions,warehouse
0,2024-03-16,2024-03-15,11092.230822,Prague_1
1,2024-03-17,2024-03-15,10675.140897,Prague_1
2,2024-03-18,2024-03-15,9298.542103,Prague_1
3,2024-03-19,2024-03-15,9389.534300,Prague_1
4,2024-03-20,2024-03-15,9185.069692,Prague_1
...,...,...,...,...
53,2024-05-12,2024-03-14,5785.501485,Budapest_1
54,2024-05-13,2024-03-14,5709.606061,Budapest_1
55,2024-05-14,2024-03-14,5654.095768,Budapest_1
56,2024-05-15,2024-03-14,5621.524334,Budapest_1


In [10]:
def get_min_datecut(data: pd.DataFrame) -> pd.DataFrame:
    data = data.reset_index()
    return data.iloc[[data.idxmin()["date_cut"]], :]

full_predictions = full_predictions\
    .groupby(["warehouse", "timestamp"])\
    .apply(lambda x: get_min_datecut(x), include_groups=False)\
    .reset_index()
full_predictions = full_predictions[["warehouse", "timestamp", "predictions"]]
full_predictions["predictions"] = round(full_predictions["predictions"])

In [11]:
full_predictions = test[["warehouse", "timestamp", "id"]].merge(full_predictions,
                                                                on=["warehouse", "timestamp"],
                                                                how="left")
full_predictions = full_predictions.rename(columns={"predictions":"orders"})[["id", "orders"]]
full_predictions

,id,orders
0,Prague_1_2024-03-16,11092.0
1,Prague_1_2024-03-17,10675.0
2,Prague_1_2024-03-18,9299.0
3,Prague_1_2024-03-19,9390.0
4,Prague_1_2024-03-20,9185.0
...,...,...
392,Budapest_1_2024-05-11,5884.0
393,Budapest_1_2024-05-12,5786.0
394,Budapest_1_2024-05-13,5710.0
395,Budapest_1_2024-05-14,5654.0


In [ ]:
output_dir = "/kaggle/working"
output_dir = "./submissions"
full_predictions.to_csv(f"{output_dir}/submission.csv", index=False)

Backtesting

In [13]:
full_predictions = pd.DataFrame()
for warehouse in tqdm(train["warehouse"].unique()):
    train_warehouse = train[train["warehouse"]==warehouse].copy(deep=True)
    test_warehouse = test[test["warehouse"]==warehouse].copy(deep=True)
    prediction_horizon = (max(test_warehouse["timestamp"]) - min(test_warehouse["timestamp"])).days + 1
    timestep = prediction_horizon
    predictions = prediction_process(train_warehouse, pd.DataFrame(columns=["timestamp","target"]),
                                      date_cut=max(train_warehouse["timestamp"] - pd.DateOffset(months=6)),
                                     prediction_horizon=prediction_horizon, timestep=timestep,
                                     freq="D")
    predictions["warehouse"] = warehouse
    full_predictions = pd.concat([full_predictions, predictions]) if not full_predictions.empty\
                                                                  else predictions
full_predictions.head()

100%|██████████| 7/7 [00:02<00:00,  3.45it/s]


,timestamp,date_cut,predictions,warehouse
0,2023-09-16,2023-09-15,8483.909631,Prague_1
1,2023-09-17,2023-09-15,8124.861249,Prague_1
2,2023-09-18,2023-09-15,8477.272501,Prague_1
3,2023-09-19,2023-09-15,8299.945019,Prague_1
4,2023-09-20,2023-09-15,8348.391700,Prague_1
...,...,...,...,...
55,2024-03-10,2024-01-14,5508.674697,Budapest_1
56,2024-03-11,2024-01-14,5362.846621,Budapest_1
57,2024-03-12,2024-01-14,5417.272452,Budapest_1
58,2024-03-13,2024-01-14,5552.575917,Budapest_1


In [ ]:
backtest_results = train[["warehouse", "timestamp", "target"]].merge(full_predictions, on=["timestamp", "warehouse"])
backtest_results = backtest_results\
    .groupby(["warehouse", "date_cut"])\
    .apply(lambda x: mean_absolute_percentage_error(x["target"], x["predictions"])).reset_index()\
    .rename(columns={0:"mape"})
backtest_results = backtest_results[["warehouse","mape"]].groupby(["warehouse"]).mean()
backtest_results = backtest_results.mean()
backtest_results

/tmp/ipykernel_93630/3843401545.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: mean_absolute_percentage_error(x["target"], x["predictions"])).reset_index()\


mape    0.110184
dtype: float64